Initial Setup



In [ ]:
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
file_list = drive.ListFile({'q': "'1iUsSO-Fo5D5c0YX9_6RgA1cy4IK_Da4-' in parents"}).GetList()
for f in file_list:
  print('title: %s, id: %s' % (f['title'], f['id']))

In [ ]:
id='1q7iH1KpSPEtd35NAOqvA35mikPhracDY'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('lastfm.csv')

Importing all the needed packages

In [ ]:
# Importing all modules related to pyspark
import pyspark
from pyspark.sql import *
from pyspark.conf import SparkConf
from pyspark import SparkContext
from pyspark.sql.functions import *
from pyspark.ml.fpm import FPGrowth

In [ ]:
# Importing all modules not related to pyspark
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('white')

from warnings import filterwarnings
filterwarnings('ignore')

Creating the spark session


In [ ]:
spark = SparkSession.builder.master('local[*]').appName('last_fm_coursework').getOrCreate()
sc = spark.sparkContext
sc

**The loading and inspection of the dataset**

In [ ]:
# The dataset is read as a spark dataframe
spark_df = spark.read.csv('/content/lastfm.csv', header=True, inferSchema=True)

# Displaying the first six rows of the dataset
print('The first six rows of the dataset')
spark_df.show(6)

# Column count
print('The number of columns or variables in this dataset: ', len(spark_df.columns))
print()

# Column names and types
print('What are the column names and types')
print(spark_df.printSchema())
print()

# Displaying the number of observations or rows
print('The number of observations in this dataset: ', spark_df.count())

Checking for duplicates

In [ ]:
spark_df_dup=spark_df.groupBy('user','artist','sex','country').count().filter("count > 1")
spark_df_dup.drop('count').show()
# The dataset has two duplicates with users 9753 and 6980.

In [ ]:
# The two duplicates are then removed.
spark_df = spark_df.dropDuplicates()

spark_df.count()
# The number of observations are now 289953 instead of 289955. This indicates that the 2 duplicates have successfully been removed.

Checking for the number of unknown artists

In [ ]:
spark_df.filter(spark_df.artist == '[unknown]').count()
# There are 553 unknown artists but they will be kept in the dataset

**Exploratory Data Analysis**

Counting distinct user, artist and country

In [ ]:
spark_df.select('user').distinct().count()  # There are 15,000 distinct users in this dataset

In [ ]:
spark_df.select('artist').distinct().count() # There are 1,004 distinct artists in this dataset

In [ ]:
spark_df.select('country').distinct().count() # There are users from 159 countries in this dataset

In [ ]:
spark_df.select('user','sex','country').distinct().show(5) # A display of distinct users by their sex and country

Counting the number of male and female users

In [ ]:
spark_df.select('user','sex','country').distinct().where(col('sex') == 'm').count() # There are 11149 male users in this dataset

In [ ]:
spark_df.select('user','sex','country').distinct().where(col('sex') == 'f').count() # There are 3851 female users in this dataset

In [ ]:
spark_df.select('user','sex','country').distinct().groupBy('sex').count().show() # A display of the count of distinct males and females

In [ ]:
spark_df.select('user','sex','country').distinct().filter(spark_df.sex == 'm').groupBy('country','sex').count().orderBy(desc('count')).show(10)
# United States had the highest number of male users who were 2,017 in number

In [ ]:
spark_df.select('user','sex','country').distinct().filter(spark_df.sex == 'f').groupBy('country','sex').count().orderBy(desc('count')).show(10)
# United States had the highest number of female users who were 888 in number

Finding the top and last countries based on the number of users

In [ ]:
spark_df.select('user','sex','country').distinct().groupBy('country').count().orderBy(desc('count')).show(10) # A display of the top 10 nationalities based on number of users.
                                                                                                              # United states is the highest country with 2905 users

In [ ]:
spark_df.select('user','sex','country').distinct().groupBy('country').count().orderBy(asc('count')).show(10) # A display of the bottom 10 countries in terms of number of users

Finding the number of countries with only one user

In [ ]:
one_user_country = spark_df.select('user','sex','country').distinct().groupBy('country').count().orderBy(asc('count')).where(col('count') == 1)
one_user_country.count()  # There are 43 countries with one user

Finding the number of countries with ten or more users

In [ ]:
more_than_ten_users_country = spark_df.select('user','sex','country').distinct().groupBy('country').count().orderBy(asc('count')).where(col('count') >= 10)
more_than_ten_users_country.count()  # There are 62 countries with ten or more users

Finding the top 10 artists listened to in the dataset

In [ ]:
spark_df.groupBy('artist').count().orderBy(desc('count')).show(10) # A list of the top 10 artists listened the most in this dataset with radiohead at the top

Finding the artists most listened to per country

In [ ]:
spark_df.select('artist', 'country').distinct().groupBy('artist').count().orderBy(desc('count')).show(10)
# The beatles and radiohead were the most listened artists per country with 96 countries listening to them

Finding the least 10 artists listened to in the dataset

In [ ]:
spark_df.groupBy('artist').count().orderBy(asc('count')).show(10) # A list of the least 10 artists listened to in this dataset with mary j. blige being the least

Finding the user's country which listened most to the top artist of the dataset (radiohead)

In [ ]:
spark_df.select('country').filter(spark_df.artist == 'radiohead').groupBy('country').count().orderBy(desc('count')).show(10)
#United States listeners contributed highest to the artist radiohead. It's not surprising because they have the highest number of users.

Finding the top artist listened to in the United States

In [ ]:
spark_df.select('artist').filter(spark_df.country == 'United States').groupBy('artist').count().orderBy(desc('count')).show(10)
# the beatles were the most listened artists in United States with 786 users

Finding the top and least 10 artists listened to by the males and females

In [ ]:
spark_df.select('artist','sex').filter(spark_df.sex == 'f').groupBy('artist').count().orderBy(desc('count')).show(10)
# With respect to the females, the artist coldplay was listened the most with 798 users

In [ ]:
spark_df.select('artist','sex').filter(spark_df.sex == 'f').groupBy('artist').count().orderBy(asc('count')).show(10)
# The artist mobb deep was the least listened among females with 2 users

In [ ]:
spark_df.select('artist','sex').filter(spark_df.sex == 'm').groupBy('artist').count().orderBy(desc('count')).show(10)
# The artist radiohead was listened the most among the males with 1982 users

In [ ]:
spark_df.select('artist','sex').filter(spark_df.sex == 'm').groupBy('artist').count().orderBy(asc('count')).show(10)
# The artist the hush sound was listened the least among males with 37 users

Finding the top ten users who listen to the highest number of artists

In [ ]:
spark_df.groupBy('user','country','sex').count().orderBy(desc('count')).show(10)
#User 17681 listens to the highest number of artists 76 to be precise and is from New Zealand and is a female. With the top ten listeners, 3 are females and 7 are males.

In [ ]:
spark_df.groupBy('country','artist').count().show(10)  # Grouping the artists listened to per country

**Visualisations**

In [ ]:
# Visualising the sex count in the dataset
sex_count = spark_df.select('user','sex','country').distinct().groupBy('sex').count().toPandas()

#plt.figure(figsize=(10,8))
sns.barplot(x='sex', y='count', data=sex_count)
plt.title('A display of number of males and females', fontsize=14);



In [ ]:
# Visualising the top 10 countries the users come from
top_10_countries = spark_df.select('user','sex','country').distinct().groupBy('country').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='country', y='count', data=top_10_countries)
plt.xticks(rotation='vertical')
plt.title('A display of top 10 countries of users', fontsize=14);
plt.savefig('Top10countries.png')


In [ ]:
# Visualising the top 10 artists listened to in the dataset
top_10_artists = spark_df.groupBy('artist').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='artist', y='count', data=top_10_artists, palette='GnBu')
plt.xticks(rotation='vertical')
plt.title('A display of top 10 artists of users', fontsize=14);


In [ ]:
# Visualising in order of 10 countries of users who listened to the top artist radiohead
radiohead_top_10_countries = spark_df.select('country').filter(spark_df.artist == 'radiohead').groupBy('country').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='country', y='count', data=radiohead_top_10_countries, palette='Wistia')
plt.xticks(rotation='vertical')
plt.title('A display of top 10 countries of users listening to radiohead', fontsize=14);

In [ ]:
# Visualising top 10 artists listened to in the country of the highest users (United States)
United_States_artists = spark_df.select('artist').filter(spark_df.country == 'United States').groupBy('artist').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='artist', y='count', data=United_States_artists, palette='crest')
plt.xticks(rotation='vertical')
plt.title('A display of top 10 artists listened to in the United States', fontsize=14);
plt.savefig('UnitedStates10.png')


In [ ]:
# A display of the top 10 listened artists by females.
top_10_female_listened_artists = spark_df.select('artist','sex').filter(spark_df.sex == 'f').groupBy('artist').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='artist', y='count', data=top_10_female_listened_artists, palette='cool')
plt.xticks(rotation='vertical')
plt.title('A display of top 10 artists listened by females', fontsize=14);

In [ ]:
# # A display of the top 10 listened artists by males.
top_10_male_listened_artists = spark_df.select('artist','sex').filter(spark_df.sex == 'm').groupBy('artist').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='artist', y='count', data=top_10_male_listened_artists, palette ='PuBu_r')
plt.xticks(rotation='vertical')
plt.title('A display of top 10 artists listened by males', fontsize=14);

In [ ]:
# A display of the top 10 listeners
top_10_listeners = spark_df.groupBy('user','country','sex').count().orderBy(desc('count')).limit(10).toPandas()

plt.figure(figsize=(10,8))
sns.barplot(x='user', y='count', data=top_10_listeners, palette='terrain')
plt.title('A display of top 10 artists listeners', fontsize=14);

**Application of FPGrowth algorithm**

In [ ]:
# First of all, the users are grouped by groupBy to obtain one row per basket (i.e. the set of artists listened by a particular user)
grouped_users = spark_df.groupBy('user').agg({'artist': 'collect_list'})
grouped_users = grouped_users.toDF('user', 'artist')

In [ ]:
grouped_users.columns # A display of the columns of artists grouped by users.

In [ ]:
grouped_users.take(10) #Observing the first ten elements of the grouped users.

Splitting the data into training and testing set to evaluate the performance of FPGrowth model

In [ ]:
train, test = grouped_users.randomSplit(weights=[0.8,0.2], seed=200) #Splitting the data into 80% training and 20% testing
print(train.count())
print(test.count())

#The training set has 11954 observations and the testing set has 3046 observations

Finding the optimum minimum support and minimum confidence

In [ ]:
# First model is built with minimum support of 0.001 and minimum confidence of 0.2
fp_growth_1 = FPGrowth(itemsCol="artist", minSupport=0.001, minConfidence=0.2)
model_1 = fp_growth_1.fit(train)

print(model_1.freqItemsets.count())
print(model_1.associationRules.count())

# The number of frequent itemsets is 282776 which is too many with respect to this dataset hence generating uninteresting associations because the minimum support is very low
# The association rules are 591791 which is too many and this is due to a low minimum support and a low minimum confidence


In [ ]:
# This next model is built by increasing the minSupport to 1 and minConfidence to 0.8

fp_growth_2 = FPGrowth(itemsCol="artist", minSupport=1, minConfidence=0.8)
model_2 = fp_growth_2.fit(train)

print(model_2.freqItemsets.count())
print(model_2.associationRules.count())

# Both the number of frequent sets and the number of associations are 0 because the minimum support was raised too high from 0.001 to 1 hence no artist qualifies as a frequent.

In [ ]:
# This next model is built by decreasing the minSupport to 0.01 and maintaining the minConfidence at 0.8

fp_growth_3 = FPGrowth(itemsCol="artist", minSupport=0.01, minConfidence=0.8)
model_3 = fp_growth_3.fit(train)

print(model_3.freqItemsets.count())
print(model_3.associationRules.count())

# There are 1666 frequent itemsets but no association rules between them
# This is because the minConfidence is very high

In [ ]:
#This next model is built by maintaining the minSupport at 0.01 and decreasing the minConfidence to 0.6

fp_growth_4 = FPGrowth(itemsCol="artist", minSupport=0.01, minConfidence=0.6)
model_4 = fp_growth_4.fit(train)

print(model_4.freqItemsets.count())
print(model_4.associationRules.count())

# There are 1666 frequent itemsets and 9 association rules between them
# The association rules are small and hence the minConfidence needs to be decreased.

In [ ]:
#This next model is built by maintaining the minSupport at 0.01 and decreasing the minConfidence to 0.3

fp_growth_5 = FPGrowth(itemsCol="artist", minSupport=0.01, minConfidence=0.3)
model_5 = fp_growth_5.fit(train)

print(model_5.freqItemsets.count())
print(model_5.associationRules.count())

print(model_5.associationRules.filter(model_5.associationRules.lift <= 1.67).count())
print(model_5.associationRules.filter(model_5.associationRules.lift > 1.67).count())


# There are 1666 frequent itemsets and 483 association rules between them.
# The lift of the association rules are greater than 1.67 indicating that the association rules are strong, useful and of high interest

In [ ]:
model_5.freqItemsets.orderBy(desc('freq')).take(10)  # A display of the 10 highest frequent itemsets

In [ ]:
print(model_5.associationRules.show(10)) # A display of 10 association rules

In [ ]:
model_5.associationRules.orderBy(desc('lift')).show(10) # A display of top 10 associatons in the descending order of their lift.

Using the test set to gauge the performance of the FPGrowth model

In [ ]:
new_output = model_5.transform(test)

Visualisation of some outputs

In [ ]:
new_output.take(10) # A display of the predictions made by the model

In [ ]:
# Calculating the number of null predictions
new_output.createOrReplaceTempView('output')
new_output_sql = spark.sql('SELECT * FROM output WHERE cardinality(prediction) == 0')
new_output_sql.count()

# There were  null predictions because there were no association rules developed for all the users listening patterns.
# 185 users out of the 3046 users had no predictions.
# They formed 6.07%
# This implies the model was able to recommend artists or music to 93.93% of users